<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/leaderboard_focus/GEMS_SUBMISSION_FACTORY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS submission factory (code rev sf.1-r1)

**One notebook, one CONFIG, run top to bottom.** Loads the frozen B0 / B1 / W2 artifacts, rebuilds every enabled candidate zone by zone, scores it with the existing official scorer, validates it, writes an official-format GeoTIFF, **reopens the written file** and checks it, then tells you which file to upload. Nothing is trained, re-inferred, tuned or uploaded.

| | |
|---|---|
| Runtime | Colab **CPU** (High-RAM recommended). No GPU. |
| Inputs (read-only) | `MyDrive/GEMS_2`: `outputs/p6_fast_track/{stack, boost/benchmark, recovery/export_parity}`, `experiments/b0_reference__d54e938a`, `experiments/b1_bg_weight__7755635b`, folds, `example_submission.tif`, M9 selection |
| Outputs | `MyDrive/GEMS_2/outputs/leaderboard_focus/submission_factory/` |
| Code | `gems_bench/leaderboard_focus/factory.py` (`Factory`) - docs `docs/leaderboard_focus/SUBMISSION_FACTORY.md` |

Per candidate, four separate verdicts: `TECHNICAL_VALIDATION`, `EXPORT_PARITY`, `OFFICIAL_FORMAT` (hard: a failure means no upload) and `GEOGRAPHIC_TARGET_MET` (internal 0.1758 research target: **reported, not blocking** in `report_only` mode). `READY_FOR_UPLOAD` means the file read back from disk passed every hard check; it is not a leaderboard prediction.

**After a disconnect:** run every cell again from the top. Finished candidates print `SKIP_CACHED`; nothing expensive is repeated.

## 1 · Configuration (the only cell you edit)

`run_mode`: `FULL` (load, evaluate, validate, write) · `EXPORT_ONLY` (skip the official recomputation; frozen-grid scores are used and flagged) · `DIAGNOSTIC` (everything except writing GeoTIFFs) · `RESUME` (= FULL with cache reuse forced). `geographic_target_mode`: `report_only` (default) or `strict` (a file below the target is written but held, `HELD_BY_STRICT_TARGET`).

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                        # pin the pushed commit SHA for a reproducible run

    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'run_mode': 'FULL',                   # FULL | EXPORT_ONLY | DIAGNOSTIC | RESUME
    'reuse_cached': True,
    'overwrite_existing': False,          # True: recompute every candidate (files of a different recipe are still never overwritten)

    'device': 'cpu',
    'train_models': False,                # True is refused

    'candidates': [
        'w2_champion',                    # a_w2_rep_frac (M1 champion, recipe 0a8692b2ab)
        'w2_crest',                       # a_w2_crest
        'w2_skeleton',                    # a_w2_skeleton (deduplicated if its raster equals the champion's)
        'b0_b1_fallback',                 # ref_m9_fallback (B0 + B1 rank fusion, no W2)
    ],

    'evaluate_official_metric': True,
    'validate_export_parity': True,

    'geographic_target': 0.1758,
    'geographic_target_mode': 'report_only',   # report_only | strict

    'export_all_technically_valid': True,
    'max_submission_files': 4,

    'save_png': True,
    'save_csv': True,
    'save_parquet': True,
    'save_geotiff': True,
    'sync_to_drive': True,

    'manual_upload_only': True,
}
PRIVATE_REPO = False
EXPECT = {'code_rev': 'sf.1-r1', 'commit': None}   # set 'commit' to the pushed SHA

## 2 · Environment and Drive

Mounts Drive, clones the pinned ref (no fallback branch), installs the package, builds the factory. The runtime must be fresh (`gems_bench` not yet imported).

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'factory_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)       # a SHA / tag must resolve, else this raises
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD

import pandas as pd
from IPython.display import Image, Markdown, display
from gems_bench.leaderboard_focus import factory as FA
assert FA.FACTORY_CODE_REV == EXPECT['code_rev'], f'code rev {FA.FACTORY_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
F = FA.Factory(CONFIG, HEAD)
RUN_ALL = F.run          # optional one-call entry point: RUN_ALL() == cells 3-6 below
OUT = Path(CONFIG['local_root']) / FA.OUTPUT_DIR
print('SHA', HEAD, '| code rev', FA.FACTORY_CODE_REV, '| mode', F.c['run_mode'], '| plan', F.plan)
print('candidates', F.c['candidates'], '| geo target', F.c['geographic_target'], F.c['geographic_target_mode'])
print('Drive outputs:', F.drive_path(''))

## 3 · Runtime self-tests (synthetic, before any export)

Raster round-trip, shape / transform, eligible mask, zone stitching, duplicate zones, budget, value range, checksum, CSV loader, candidate isolation, recipe identity, status logic, ranking, config guards. A failed **critical** test disables every export of this run (the preflight then reports it as BLOCKED).

In [ ]:
ST = F.selftest()
display(ST[['test', 'critical', 'status', 'detail']])

## 4 · Read-only artifact preflight  `[1/6]`

Existence, CSV schema / rows, B0 / B1 experiment ids, raster CRS / transform vs the template, W2 maps, candidate grid cells, earlier output files. Each row is PASS / WARNING / BLOCKED; a shared BLOCKED row blocks every candidate, a candidate row only that candidate (a missing W2 map never blocks the B0 / B1 fallback). Writes `diagnostics/preflight.{csv,json}` and `reports/PREFLIGHT.md`.

In [ ]:
PF = F.preflight()
display(PF[PF.status != 'PASS'][['status', 'scope', 'candidate', 'check', 'detail', 'action']])

## 5 · Frozen models  `[2/6]`

Attaches the read-only engine on the CPU, recomputes the shared nested candidate table from the frozen boost-A grid (same folds, eligible area, scorer and protocol for every candidate), checks the official template and the geographic fold geometry (fold assignment, duplicated / uncovered pixels).

In [ ]:
OK = F.attach()
if OK:
    display(F.C[[c for c in ('candidate_id', 'status', 'eligible', 'disc_dti', 'disc_worst', 'geo_dti', 'deploy',
                             'deploy_cell') if c in F.C]])

## 6 · Candidates: recovery → official evaluation → technical validation → GeoTIFF → read-back  `[3-6/6]`

Each candidate runs independently and is saved (and pushed to Drive) as soon as it finishes; a failure in one never touches the others. A candidate whose validated raster equals an earlier one is recorded as `DUPLICATE` (no redundant file).

In [ ]:
for cid in F.c['candidates']:
    if not OK and not F.block.get(cid):
        F.block[cid] = ['frozen models stage did not complete (diagnostics/runtime_errors.csv)']
    F.candidate(cid)
F.finish()

## 7 · Submission ranking, manifest and figures

Read from the persisted files (`submissions/submission_manifest.csv`, `metrics/`, `figures/`). Upload priority: `PRIMARY` (predeclared rule: p6g-eligible first, then discovery DTI, then worst discovery zone), at most one `ALTERNATIVE` (higher single-policy geographic DTI than the primary: it tests the discovery / geographic trade-off on the leaderboard), `OPTIONAL`, `DO_NOT_UPLOAD`.

In [ ]:
M = pd.read_csv(OUT / 'submissions/submission_manifest.csv', dtype={'recipe_id': str})
display(M[['candidate', 'status', 'upload_priority', 'recommendation_rank', 'file', 'TECHNICAL_VALIDATION',
           'EXPORT_PARITY', 'OFFICIAL_FORMAT', 'GEOGRAPHIC_TARGET_MET', 'disc_dti', 'geo_dti', 'duplicate_of',
           'first_reason']])
display(pd.read_csv(OUT / 'metrics/per_zone_scores.csv') if (OUT / 'metrics/per_zone_scores.csv').stat().st_size
        else 'no per-zone scores')
for fig in ('candidate_comparison', 'discovery_geographic_tradeoff', 'per_zone_dti', 'prediction_coverage'):
    p = OUT / 'figures' / f'{fig}.png'
    if p.exists():
        display(Image(filename=str(p)))

## 8 · Final execution report: which file to upload

In [ ]:
display(Markdown((OUT / 'submissions/README_UPLOAD.md').read_text(encoding='utf-8')))
display(Markdown((OUT / 'reports/NEXT_ACTION.md').read_text(encoding='utf-8')))
print(open(OUT / 'submissions/submission_sha256.txt').read() or 'no READY file')

## 9 · Manual leaderboard result (only after you uploaded a READY file by hand)

Fill `RESULT` and run. Refused when the candidate / recipe / checksum does not match a READY file, or the id / file was already recorded. Public scores are history only: they never change any offline result or selection.

In [ ]:
RESULT = {
    'submission_id': '',          # DrivenData submission id
    'candidate': '',              # e.g. 'w2_champion'
    'recipe_id': '',              # from submission_manifest.csv
    'file_sha256': '',            # sha256sum of the uploaded file
    'public_score': None,
    'submission_date': '',        # YYYY-MM-DD
    'notes': '',
}
if RESULT['submission_id']:
    print(F.record_result(RESULT))
    display(pd.read_csv(OUT / 'leaderboard/leaderboard_results.csv'))
else:
    print('no result entered (fill RESULT after a manual upload)')